# Customer Segmentation using K-Means Clustering
This notebook cleans customer data, explores purchasing behavior, finds an optimal K using the Elbow Method and Silhouette Score, applies K-Means, and summarizes customer segments.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv('../dataset/customer_data.csv')
df.head()

In [ ]:
df.info()
df.isna().sum()
df.describe(include='all')

In [ ]:
features = ['AnnualIncome_k','SpendingScore','PurchaseFrequency','AvgOrderValue']
scaler = StandardScaler()
X = scaler.fit_transform(df[features])

In [ ]:
inertias, silhouettes = [], []
for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X, labels))

plt.plot(range(2,9), inertias, marker='o')
plt.xlabel('K'); plt.ylabel('Inertia'); plt.title('Elbow Method'); plt.show()

plt.plot(range(2,9), silhouettes, marker='o')
plt.xlabel('K'); plt.ylabel('Silhouette Score'); plt.title('Silhouette Score'); plt.show()

In [ ]:
best_k = 3
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X)
df.groupby('Cluster')[features].mean().round(2)

In [ ]:
profiles = df.groupby('Cluster')[features].mean()
score = ((profiles['AnnualIncome_k']-profiles['AnnualIncome_k'].mean())/profiles['AnnualIncome_k'].std() +
         (profiles['SpendingScore']-profiles['SpendingScore'].mean())/profiles['SpendingScore'].std())
ordered = score.sort_values().index.tolist()
names = ['Low Value','Developing','Potential High Value','Premium']
name_map = {cl: names[i] if i < len(names) else f'Segment {i+1}' for i, cl in enumerate(ordered)}
df['Segment'] = df['Cluster'].map(name_map)
df.to_csv('../dataset/customer_segments.csv', index=False)
df.head()

In [ ]:
for seg, g in df.groupby('Segment'):
    plt.scatter(g['AnnualIncome_k'], g['SpendingScore'], s=30, alpha=.7, label=seg)
plt.xlabel('Annual Income (₹ lakh)')
plt.ylabel('Spending Score')
plt.title('Customer Segments')
plt.legend()
plt.show()

## Business Insights
- Premium customers can be targeted with loyalty programs and exclusive offers.
- Potential High Value customers can receive cross-sell and upsell campaigns.
- Developing customers may respond to personalized discounts and product recommendations.
- Low Value customers can be addressed with low-cost retention campaigns.